# 🎰 Aula 9 — Modelos Discretos na prática: a matemática das apostas esportivas

Este notebook **não tem nota nem entrega**, e você **não precisa saber programar**. É só executar as células, na ordem, de cima para baixo, e ler os textos entre elas.

A ideia: na Aula 9 vimos três modelos — **uniforme discreta**, **Bernoulli** e **geométrica** — e calculamos tudo com as fórmulas, à mão. Aqui vamos usar o computador para **simular** situações de um app de apostas (as famosas *bets*) e conferir que as fórmulas da aula acertam o que acontece na prática.

Cada seção segue sempre a mesma ordem:
1. **A situação e a pergunta** (o enunciado, como num exercício);
2. **A ferramenta da aula** que responde a pergunta;
3. **A resposta pela fórmula**;
4. **A simulação** — o computador repete a situação milhares de vezes;
5. **Como ler o resultado**.

> ⚠️ **Aviso:** conteúdo **exclusivamente educacional**. As probabilidades e *odds* são **fictícias**, escolhidas só para ilustrar os modelos. Não é incentivo a apostas — pelo contrário: no final você vai ver, com números, por que quem aposta repetidamente tende a perder dinheiro.
>
> Notebook elaborado com auxílio de IA (Claude) e conferido pelo professor.

## Como executar

- Cada **célula de código** (as caixas cinza) tem um botão **▶** do lado esquerdo. Clique nele, ou clique na célula e aperte **Shift + Enter**.
- Execute **uma de cada vez, na ordem**: algumas células usam o resultado das anteriores.
- Na primeira vez, o Colab pode avisar que o notebook "não foi criado pelo Google". Clique em **Executar mesmo assim**.
- Se preferir, use o menu **Ambiente de execução → Executar tudo** e depois só leia, de cima para baixo.
- Detalhe: o Python escreve números com **ponto** no lugar da vírgula (`5.5` = 5,5).

## 0. Preparando as ferramentas

Antes de começar, a célula abaixo faz três coisas:

- **Carrega o `numpy`**, uma "calculadora" do Python que faz contas com milhares de números de uma vez e sabe **sortear** números aleatórios.
- **Carrega o `matplotlib`**, que desenha os gráficos.
- **Cria o nosso sorteador** (`rng`). É ele que vai "girar a roleta" e "decidir quem ganha a aposta" nas simulações. O número **42** é a **semente**: ele faz o sorteio sair **sempre igual**, toda vez que alguém executa o notebook. Por isso os números que aparecerem na sua tela serão os mesmos citados nos textos. (Qualquer outro número serviria; 42 é só uma escolha.)

**O que é simular?** É usar o computador para repetir um experimento aleatório milhares de vezes, como se estivéssemos fazendo de verdade. Chamamos de:
- **empírico** — o que sai da simulação (o que "aconteceu");
- **teórico** — o que as fórmulas da aula dizem que deveria acontecer.

Se as fórmulas estiverem certas, os dois devem ficar **bem próximos**.

In [ ]:
import numpy as np                  # calculadora de números e sorteios
import matplotlib.pyplot as plt     # desenhador de gráficos

rng = np.random.default_rng(42)     # nosso sorteador, com semente 42 (sorteios sempre iguais)
print("Ferramentas prontas!")

## 1. Uniforme discreta — a roleta de bônus do app

### A situação e a pergunta
O app oferece uma **roleta de bônus** com 10 casas, numeradas de 1 a 10, **todas com a mesma chance** de sair. O número sorteado é quantos pontos de bônus você ganha. Seja $X$ o número sorteado.

> **Pergunta:** em média, quantos pontos você ganha por giro? E quanto os resultados variam em torno dessa média?

### A ferramenta da aula
$X$ assume os valores $1, 2, \dots, 10$, todos com probabilidade $1/10$: é uma **uniforme discreta** com $n = 10$ — o mesmo caso do exemplo do dado d8 da aula. As fórmulas são:

$$E(X) = \frac{n+1}{2} \qquad\qquad Var(X) = \frac{n^2-1}{12}$$

### A resposta pela fórmula
A célula abaixo só aplica as fórmulas com $n = 10$ — ainda não tem sorteio nenhum.

In [ ]:
k = 10                              # número de casas da roleta (resultados de 1 a 10)

media_teorica = (k + 1) / 2         # E(X) = (n+1)/2
var_teorica = (k**2 - 1) / 12       # Var(X) = (n²-1)/12

print(f"E(X) pela fórmula   = {media_teorica}")
print(f"Var(X) pela fórmula = {var_teorica}")

**Como ler:** a fórmula diz que você ganha, em média, **5,5 pontos por giro** — um valor que a roleta nem tem! A esperança é uma média de longo prazo, não um resultado possível. A variância, **8,25**, mede o quanto os giros se espalham em torno de 5,5.

### A simulação
Agora vamos **girar a roleta 200 mil vezes** com o sorteador, calcular a média e a variância desses 200 mil resultados (como se fossem dados coletados de verdade) e comparar com a fórmula.

In [ ]:
n_giros = 200_000                                # quantas vezes vamos girar a roleta

giros = rng.integers(1, k + 1, size=n_giros)     # gira 200 mil vezes: cada giro sorteia um número de 1 a 10, todos com a mesma chance

media_empirica = giros.mean()                    # média dos 200 mil resultados
var_empirica = giros.var()                       # variância dos 200 mil resultados

print(f"Primeiros 10 giros: {giros[:10]}")
print()
print(f"E(X):   simulação = {media_empirica:.4f}   |   fórmula = {media_teorica:.4f}")
print(f"Var(X): simulação = {var_empirica:.4f}   |   fórmula = {var_teorica:.4f}")

**Como ler:** a primeira linha mostra os 10 primeiros giros, só para você ver que são números "soltos", de 1 a 10. As duas linhas de baixo comparam a simulação com a fórmula: a média simulada deu cerca de **5,506** (a fórmula diz 5,5) e a variância cerca de **8,239** (a fórmula diz 8,25).

Não bate **exatamente** porque é sorteio — mas bate **muito de perto**. Quanto mais giros, mais perto a simulação fica da fórmula.

### O gráfico
A célula abaixo desenha um **histograma** dos 200 mil giros: uma barra para cada resultado (1 a 10), com altura igual a **quantas vezes aquele resultado saiu**.

In [ ]:
plt.figure(figsize=(6, 4))                                              # abre uma "tela" em branco para o gráfico
plt.hist(giros, bins=np.arange(0.5, k + 1.5, 1), rwidth=0.8,            # desenha uma barra para cada resultado de 1 a 10,
         color="#8A3B3F")                                               # com altura = quantas vezes ele saiu
plt.title("Roleta de bônus: 200 mil giros")                             # título
plt.xlabel("Resultado da roleta")                                       # nome do eixo horizontal
plt.ylabel("Quantas vezes saiu")                                        # nome do eixo vertical
plt.show()                                                              # mostra o gráfico

**Como ler:** as 10 barras têm **praticamente a mesma altura** — cerca de 20 mil cada (200 mil giros ÷ 10 casas). Essa é a "cara" da uniforme discreta: nenhum resultado é favorecido.

## 2. Bernoulli — uma única aposta

### A situação e a pergunta
Você aposta **R\$ 1,00** em "Time A vence". A casa oferece **odd 2,00**, o que significa:
- se o Time A **vencer**, você recebe R\$ 2,00 — isto é, **lucro de R\$ 1,00**;
- se o Time A **não vencer**, você **perde o R\$ 1,00** apostado.

Suponha que a chance **real** de o Time A vencer seja de **45%**.

> **Pergunta:** em média, você ganha ou perde dinheiro com essa aposta? Quanto?

### A ferramenta da aula
É **uma única tentativa** com **dois resultados** (vence / não vence): uma **Bernoulli** com $p = 0{,}45$. Seja $X = 1$ se o Time A vencer e $X = 0$ caso contrário. Da aula, $E(X) = p$.

Mas a pergunta é sobre o **lucro**, que vale $+1$ quando $X = 1$ e $-1$ quando $X = 0$. Aplicando a definição de esperança (valor × probabilidade, somados):

$$E(\text{lucro}) = (+1)\cdot p + (-1)\cdot(1-p) = 0{,}45 - 0{,}55 = -0{,}10$$

### De onde vem esse prejuízo? A odd esconde uma probabilidade
Uma odd de 2,00 só seria uma aposta "justa" (lucro médio zero) se a chance de vitória fosse $1/2{,}00 = 0{,}50$. Essa é a **probabilidade implícita** na odd. Como a chance real é 0,45, **menor** que 0,50, a casa paga menos do que deveria. Essa diferença é a **margem da casa** (no jargão das apostas, *overround*): é ela que garante o lucro de quem opera a bet.

In [ ]:
p_real = 0.45                                         # chance real de o Time A vencer
odd = 2.00                                            # odd oferecida pela casa

prob_implicita = 1 / odd                              # probabilidade que a odd "sugere"
lucro_esperado_teorico = p_real * (odd - 1) - (1 - p_real) * 1   # E(lucro) = (+1)·p + (-1)·(1-p)

print(f"Chance real de vitória:             {p_real:.2f}")
print(f"Probabilidade implícita na odd:     {prob_implicita:.2f}")
print(f"Lucro médio por aposta (fórmula):   R$ {lucro_esperado_teorico:.2f}")

**Como ler:** a fórmula diz que você **perde, em média, R\$ 0,10 a cada R\$ 1,00 apostado**, porque a chance real (0,45) é menor que a implícita na odd (0,50).

### A simulação
Agora vamos simular **200 mil apostas** de R\$ 1,00, uma independente da outra. Para cada aposta, o sorteador decide se o Time A venceu (com chance 45%); depois somamos o lucro e dividimos pelo número de apostas.

In [ ]:
n_apostas = 200_000                                        # quantas apostas vamos simular

vitorias = rng.random(n_apostas) < p_real                  # para cada aposta, sorteia se o Time A venceu (chance de 45%)
lucro_por_aposta = np.where(vitorias, odd - 1, -1)         # lucro de cada aposta: +1 se venceu, -1 se perdeu

print(f"Em quantas apostas o Time A venceu: {vitorias.mean():.2%}")
print(f"Lucro médio por aposta (simulação): R$ {lucro_por_aposta.mean():.4f}")
print(f"Lucro médio por aposta (fórmula):   R$ {lucro_esperado_teorico:.4f}")

**Como ler:** o Time A venceu em cerca de **45%** das apostas simuladas, como esperado. Mesmo assim, o lucro médio saiu por volta de **−R\$ 0,099** — praticamente os −R\$ 0,10 da fórmula.

Ou seja: **ganhar quase metade das vezes não basta.** Como a odd paga menos do que a chance real justificaria, cada aposta, em média, tira 10 centavos do seu bolso.

## 3. Geométrica — quantas apostas até a primeira vitória?

### A situação e a pergunta
Você decide fazer apostas desse mesmo tipo, **uma atrás da outra**, até ganhar **pela primeira vez**. Cada aposta tem sempre 45% de chance de vitória, e uma não influencia a outra. Seja $X$ o número de apostas feitas até a primeira vitória (**contando** a aposta vencedora).

> **Perguntas:** em média, quantas apostas você vai precisar fazer? E qual é a chance de perder as 3 primeiras seguidas, ou seja, de precisar de **mais de 3** apostas?

### A ferramenta da aula
Tentativas independentes, sempre com a mesma chance de sucesso, repetidas **até o primeiro sucesso**: é a **geométrica** com $p = 0{,}45$, na versão da aula, $P(X = k) = (1-p)^{k-1}\,p$. Da aula:

$$E(X) = \frac{1}{p} \qquad\qquad Var(X) = \frac{1-p}{p^2}$$

E precisar de mais de 3 apostas é o mesmo que **perder as 3 primeiras**: $P(X > 3) = (1-p)^3$.

In [ ]:
media_teorica_geo = 1 / p_real                    # E(X) = 1/p
var_teorica_geo = (1 - p_real) / p_real**2        # Var(X) = (1-p)/p²
prob_mais_de_3 = (1 - p_real)**3                  # P(X > 3) = perder as 3 primeiras

print(f"E(X) pela fórmula     = {media_teorica_geo:.4f}")
print(f"Var(X) pela fórmula   = {var_teorica_geo:.4f}")
print(f"P(X > 3) pela fórmula = {prob_mais_de_3:.4f}")

**Como ler:** em média você precisa de cerca de **2,22 apostas** até ganhar a primeira. E existe cerca de **16,6%** de chance de perder as 3 primeiras seguidas — mais ou menos 1 apostador em cada 6.

### A simulação
Agora vamos simular **50 mil apostadores**. Cada um aposta até ganhar pela primeira vez, e anotamos quantas apostas ele precisou fazer. O comando `rng.geometric` faz exatamente isso: para cada apostador, devolve o número de tentativas até o primeiro sucesso.

In [ ]:
n_apostadores = 50_000                                             # quantos apostadores vamos simular

tentativas = rng.geometric(p_real, size=n_apostadores)            # para cada apostador: quantas apostas até a 1ª vitória

print(f"Primeiros 10 apostadores precisaram de: {tentativas[:10]} apostas")
print()
print(f"E(X):     simulação = {tentativas.mean():.4f}   |   fórmula = {media_teorica_geo:.4f}")
print(f"Var(X):   simulação = {tentativas.var():.4f}   |   fórmula = {var_teorica_geo:.4f}")
print(f"P(X > 3): simulação = {(tentativas > 3).mean():.4f}   |   fórmula = {prob_mais_de_3:.4f}")
print(f"Maior número de apostas até ganhar: {tentativas.max()}")

**Como ler:** a média simulada ficou em cerca de **2,220** (fórmula: 2,222), e a proporção de apostadores que precisaram de mais de 3 apostas, cerca de **16,6%**, igual à fórmula. Repare na última linha: algum apostador azarado precisou de **17 apostas** até ganhar a primeira — acontece, mesmo com 45% de chance a cada vez.

### O gráfico
A célula abaixo desenha, para cada número de apostas (1, 2, 3, …), **quantos apostadores** precisaram exatamente daquele número até a primeira vitória.

In [ ]:
plt.figure(figsize=(6, 4))                                                  # abre uma "tela" em branco
plt.hist(tentativas, bins=np.arange(0.5, 16.5, 1), rwidth=0.8,              # uma barra para cada número de apostas (1 a 15),
         color="#8A3B3F")                                                   # com altura = quantos apostadores precisaram disso
plt.title("Quantas apostas até a primeira vitória (p = 0,45)")              # título
plt.xlabel("Número de apostas até a 1ª vitória")                            # eixo horizontal
plt.ylabel("Quantos apostadores")                                           # eixo vertical
plt.show()                                                                  # mostra o gráfico

**Como ler:** a barra mais alta é a do **1** — ganhar logo na primeira aposta é o resultado mais provável (45%). Cada barra seguinte tem cerca de **55%** da altura da anterior: para chegar a $k$ apostas, é preciso perder mais uma vez, e perder tem chance 0,55. É essa queda "em escada", sempre pelo mesmo fator, que dá o nome **geométrica** à distribuição.

## 4. Juntando tudo — e se eu apostar muitas vezes?

### A situação e a pergunta
Agora você faz **200 apostas seguidas** de R\$ 1,00, sempre com a mesma odd e a mesma chance.

> **Pergunta:** no final das 200 apostas, quanto você espera ter ganhado ou perdido?

### A ferramenta da aula
Vimos na Seção 2 que **cada** aposta tem lucro médio de −R\$ 0,10. A média de uma soma é a soma das médias, então, em 200 apostas:

$$E(\text{lucro total}) = 200 \times (-0{,}10) = -\text{R\$ } 20{,}00$$

### A simulação
Vamos simular **2 mil apostadores**, cada um fazendo 200 apostas. Para cada um, acompanhamos o **saldo acumulado** aposta após aposta (quanto ele já ganhou ou perdeu até ali). No gráfico aparece a **média** desses 2 mil saldos, comparada com a reta da fórmula.

In [ ]:
n_apostas_seq = 200                                                  # apostas feitas por cada apostador
n_trajetorias = 2_000                                                # quantos apostadores

resultados = rng.random((n_trajetorias, n_apostas_seq)) < p_real     # sorteia vitória/derrota de todas as apostas de todos
lucros = np.where(resultados, odd - 1, -1)                           # lucro de cada aposta: +1 ou -1
saldo = np.cumsum(lucros, axis=1)                                    # saldo acumulado de cada apostador, aposta após aposta
saldo_medio = saldo.mean(axis=0)                                     # média dos 2 mil saldos em cada momento

saldo_esperado = lucro_esperado_teorico * np.arange(1, n_apostas_seq + 1)   # a reta da fórmula: -0,10 por aposta

plt.figure(figsize=(7, 4))                                                              # abre a "tela"
plt.plot(saldo_medio, label="Saldo médio dos 2 mil apostadores (simulação)", color="#8A3B3F")   # linha da simulação
plt.plot(saldo_esperado, "--", label="Saldo esperado (fórmula)", color="black")        # reta da fórmula, tracejada
plt.axhline(0, color="gray", linewidth=0.8)                                             # linha do zero (nem ganhou nem perdeu)
plt.title("Saldo acumulado ao longo de 200 apostas de R$ 1,00")
plt.xlabel("Número de apostas feitas")
plt.ylabel("Saldo acumulado (R$)")
plt.legend()
plt.show()

print(f"Saldo médio após 200 apostas (simulação): R$ {saldo_medio[-1]:.2f}")
print(f"Saldo esperado após 200 apostas (fórmula): R$ {saldo_esperado[-1]:.2f}")
print(f"Apostadores que terminaram no prejuízo: {(saldo[:, -1] < 0).mean():.1%}")

**Como ler:** a linha da simulação **desce o tempo todo**, colada na reta tracejada da fórmula: a cada aposta, o saldo médio cai mais 10 centavos. Depois de 200 apostas, o saldo médio ficou em cerca de **−R\$ 19,82** (fórmula: −R\$ 20,00).

A última linha é a mais importante: cerca de **92% dos apostadores terminaram no prejuízo**. Alguns poucos saíram ganhando — é essa minoria que aparece contando vantagem —, mas a esperança negativa garante que a grande maioria perde. E, quanto mais se aposta, mais o resultado de cada um se aproxima da perda média.

---

## 🤔 Pergunta para pensar

Na Seção 2, o lucro médio foi negativo porque a chance real (0,45) era menor que a probabilidade implícita na odd (0,50). Volte à **primeira célula de código da Seção 2**, troque `p_real = 0.45` por `p_real = 0.50` e execute de novo; depois tente `p_real = 0.55`. O que acontece com o lucro esperado em cada caso? Por que você acha que as casas de apostas raramente oferecem odds em que a chance real é **maior** que a implícita?

*(Se mudar algum valor, os números das seções seguintes também mudam. Para voltar ao original, volte o valor para `0.45` e use **Ambiente de execução → Reiniciar sessão e executar tudo**.)*

---

## Para fechar

Você acabou de ver os três modelos da Aula 9 funcionando numa situação real:

- **Uniforme discreta:** todos os resultados igualmente prováveis (a roleta de bônus);
- **Bernoulli:** uma única tentativa, dois resultados (uma aposta);
- **Geométrica:** repetir tentativas independentes até o primeiro sucesso (apostar até ganhar).

E, juntando os três, a mesma ideia de **esperança** que usamos em sala explica por que apostar repetidamente leva ao prejuízo.

Na próxima aula vem a **binomial**, que responde outra pergunta natural: em $n$ apostas, **quantas** eu ganho?